In [17]:
# ==========================================
# CELL 1: IMPORTS
# ==========================================
import numpy as np
import pandas as pd 
import plotly.graph_objects as go
import plotly.express as px
from scipy.interpolate import PchipInterpolator # Better than CubicSpline for parametric corners
import ipywidgets as widgets
from IPython.display import display, clear_output

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [18]:
# ==========================================
# CELL 2: DATA LOADING AND PROCESSING
# ==========================================
print("Loading dual massive CSVs... this may take a minute depending on your RAM...")

# Load Both CSVs generated by the unified OCEAN script
df_nmos_raw = pd.read_csv('./nmos_LUT.csv')
df_pmos_raw = pd.read_csv('./pmos_LUT.csv')

# --- Process NMOS ---
df_nmos_raw['gm/Id'] = df_nmos_raw['GM'] / df_nmos_raw['ID']
df_nmos_raw['gm/gds'] = df_nmos_raw['GM'] / df_nmos_raw['GDS']
df_nmos_raw['gm/cgg'] = df_nmos_raw['GM'] / df_nmos_raw['CGG']
df_nmos_raw['Id/W'] = df_nmos_raw['ID'] / df_nmos_raw['W']   # DYNAMIC WIDTH SCALING
df_nmos_raw['gm/W'] = df_nmos_raw['GM'] / df_nmos_raw['W']   # DYNAMIC WIDTH SCALING
df_nmos_raw['gds/Id'] = df_nmos_raw['GDS'] / df_nmos_raw['ID']
df_nmos_raw['Vov'] = df_nmos_raw['VGS'] - df_nmos_raw['VTH']

# --- Process PMOS ---
df_pmos_raw['gm/Id'] = df_pmos_raw['GM'] / df_pmos_raw['ID']
df_pmos_raw['gm/gds'] = df_pmos_raw['GM'] / df_pmos_raw['GDS']
df_pmos_raw['gm/cgg'] = df_pmos_raw['GM'] / df_pmos_raw['CGG']
df_pmos_raw['Id/W'] = df_pmos_raw['ID'] / df_pmos_raw['W']   # DYNAMIC WIDTH SCALING
df_pmos_raw['gm/W'] = df_pmos_raw['GM'] / df_pmos_raw['W']   # DYNAMIC WIDTH SCALING
df_pmos_raw['gds/Id'] = df_pmos_raw['GDS'] / df_pmos_raw['ID']
df_pmos_raw['Vov'] = df_pmos_raw['VGS'].abs() - df_pmos_raw['VTH'].abs()

# ==========================================
# DYNAMIC CHANNEL LENGTH DETECTION
# ==========================================
unique_l_vals = sorted(df_nmos_raw['L'].unique())

l_mapping = {}
for l_val in unique_l_vals:
    if l_val < 0.99e-6:
        label = f"{round(l_val * 1e9, 2):g}nm"
    else:
        label = f"{round(l_val * 1e6, 2):g}um"
    l_mapping[l_val] = label

lengths = list(l_mapping.values())

# Robust default selection mirroring the MATLAB logic (Selects up to 6 evenly spaced lengths)
num_lengths = len(lengths)
num_defaults = min(6, num_lengths)
default_indices = np.round(np.linspace(0, num_lengths - 1, num_defaults)).astype(int)
default_selected_lengths = [lengths[i] for i in default_indices]

nmos_grouped = {}
pmos_grouped = {}

# DECIMATION FACTOR (Adjust this if the plot is too slow)
plot_decimation = 10 

# Map NMOS
for l_val, group in df_nmos_raw.groupby('L'):
    mapped_label = next((label for length, label in l_mapping.items() if np.isclose(l_val, length, rtol=1e-5)), None)
    if mapped_label:
        nmos_grouped[mapped_label] = group[['gm/Id', 'gm/gds', 'Id/W', 'gm/W', 'gm/cgg', 'gds/Id', 'Vov']].iloc[::plot_decimation, :].reset_index(drop=True)

# Map PMOS
for l_val, group in df_pmos_raw.groupby('L'):
    mapped_label = next((label for length, label in l_mapping.items() if np.isclose(l_val, length, rtol=1e-5)), None)
    if mapped_label:
        pmos_grouped[mapped_label] = group[['gm/Id', 'gm/gds', 'Id/W', 'gm/W', 'gm/cgg', 'gds/Id', 'Vov']].iloc[::plot_decimation, :].reset_index(drop=True)

idc_raw = df_nmos_raw[df_nmos_raw['L'] == df_nmos_raw['L'].unique()[0]]['ID'].iloc[::plot_decimation].reset_index(drop=True)
idc = idc_raw.apply(
    lambda x: f"{x*1e3:.1f}m" if x >= 1e-3 else 
              f"{x*1e6:.1f}u" if x >= 1e-6 else 
              f"{x*1e9:.1f}n" if x >= 1e-9 else 
              f"{x*1e12:.1f}p"
)

available_params = ['gm/Id', 'gm/gds', 'Id/W', 'gm/W', 'gm/cgg', 'gds/Id', 'Vov']

print(f"Success! Detected {len(lengths)} unique channel lengths. Ready to plot.")

Loading dual massive CSVs... this may take a minute depending on your RAM...
Success! Detected 17 unique channel lengths. Ready to plot.


In [19]:
# ==========================================
# CELL 3: PLOTTING ENGINE (With Interpolation)
# ==========================================
color_palette = px.colors.qualitative.Alphabet
length_color_map = {length: color_palette[i % len(color_palette)] for i, length in enumerate(lengths)}

def create_interactive_plot(transistor_type='nmos', x_param='gm/Id', y_param='gm/gds', selected_l_values=None, show_markers=False,
                          vline_x=None, hline_y=None, enable_interpolation=True):
    
    if selected_l_values is None or len(selected_l_values) == 0:
        selected_l_values = default_selected_lengths
    
    fig = go.Figure()
    
    if transistor_type == 'nmos':
        data_grouped = nmos_grouped
        transistor_name = 'NMOS'
    else:
        data_grouped = pmos_grouped
        transistor_name = 'PMOS'
    
    all_traces_data = []
    
    for i, l_val in enumerate(selected_l_values):
        if l_val in data_grouped:
            data = data_grouped[l_val]
            
            x_data = data[x_param].values
            y_data = data[y_param].values
            
            mask = ~np.isnan(x_data) & ~np.isnan(y_data)
            x_clean = x_data[mask]
            y_clean = y_data[mask]
            
            x_dense, y_dense = None, None
            x_unique, y_unique = x_clean, y_clean
            
            if enable_interpolation and len(x_clean) > 3:
                try:
                    # [THE ULTIMATE FIX: ARC-LENGTH PARAMETRIZATION]
                    # Never sort the arrays. We calculate the normalized distance between consecutive points.
                    scale_x = np.ptp(x_clean) + 1e-12
                    scale_y = np.ptp(y_clean) + 1e-12
                    
                    dx_norm = np.diff(x_clean) / scale_x
                    dy_norm = np.diff(y_clean) / scale_y
                    
                    dist = np.sqrt(dx_norm**2 + dy_norm**2)
                    t = np.concatenate(([0], np.cumsum(dist)))
                    
                    # Remove duplicated identical points to prevent NaN errors in interpolation
                    t_unique, uniq_idx = np.unique(t, return_index=True)
                    x_unique = x_clean[uniq_idx]
                    y_unique = y_clean[uniq_idx]
                    
                    if len(t_unique) > 3:
                        cs_x = PchipInterpolator(t_unique, x_unique)
                        cs_y = PchipInterpolator(t_unique, y_unique)
                        
                        t_dense = np.linspace(t_unique.min(), t_unique.max(), min(1000, len(t_unique) * 10))
                        x_dense = cs_x(t_dense)
                        y_dense = cs_y(t_dense)
                except Exception as e:
                    x_dense, y_dense = None, None
            
            mode = 'lines+markers' if show_markers else 'lines'
            marker_size = 6 if show_markers else 0
            color = length_color_map.get(l_val, '#1f77b4')
            
            trace_data = {
                'x': x_clean,
                'y': y_clean,
                'x_dense': x_dense,
                'y_dense': y_dense,
                'color': color,
                'name': f'L={l_val}'
            }
            all_traces_data.append(trace_data)
            
            if enable_interpolation and x_dense is not None:
                fig.add_trace(go.Scatter(
                    x=x_dense, y=y_dense, mode='lines', name=f'L={l_val}',
                    line=dict(width=2, color=color),
                    hovertemplate=(f'<b>L={l_val}</b><br>{x_param}: %{{x:.6f}}<br>{y_param}: %{{y:.6f}}<br><span style="color: red">Interpolated</span><extra></extra>'),
                    showlegend=True
                ))
                
                if show_markers:
                    fig.add_trace(go.Scatter(
                        x=x_unique, y=y_unique, mode='markers',
                        marker=dict(size=6, color=color, symbol='circle'),
                        hovertemplate=(f'<b>L={l_val}</b><br>{x_param}: %{{x:.6f}}<br>{y_param}: %{{y:.6f}}<br><span style="color: blue">Original</span><extra></extra>'),
                        showlegend=False, name=f'L={l_val} points'
                    ))
            else:
                fig.add_trace(go.Scatter(
                    x=x_clean, y=y_clean, mode=mode, name=f'L={l_val}',
                    line=dict(width=2, color=color), marker=dict(size=marker_size, color=color),
                    hovertemplate=f'<b>L={l_val}</b><br>{x_param}: %{{x:.6f}}<br>{y_param}: %{{y:.6f}}<br>Id: %{{text}}<extra></extra>',
                    text=[f'{val}' for val in idc.iloc[mask]] if hasattr(idc, 'iloc') else [],
                    showlegend=True
                ))

    # Guide Lines Intersections based on Arrays (solves multi-root parametric problems)
    if vline_x is not None:
        all_y = [y for t in all_traces_data for y in (t['y_dense'] if t['y_dense'] is not None else t['y'])]
        y_min, y_max = (min(all_y), max(all_y)) if all_y else (0, 1)
        
        fig.add_trace(go.Scatter(
            x=[vline_x, vline_x], y=[y_min, y_max], mode='lines', name='Vertical Guide',
            line=dict(color='rgba(128, 128, 128, 0.7)', width=1.5, dash='dot'),
            showlegend=False, hoverinfo='skip'
        ))
        
        for trace_data in all_traces_data:
            x_curve = trace_data['x_dense'] if trace_data['x_dense'] is not None else trace_data['x']
            y_curve = trace_data['y_dense'] if trace_data['y_dense'] is not None else trace_data['y']
            
            if len(x_curve) > 0 and x_curve.min() <= vline_x <= x_curve.max():
                idx = np.argmin(np.abs(x_curve - vline_x))
                if abs(x_curve[idx] - vline_x) < (x_curve.max() - x_curve.min()) * 0.05:
                    fig.add_trace(go.Scatter(
                        x=[vline_x], y=[y_curve[idx]], mode='markers',
                        marker=dict(size=12, color=trace_data['color'], symbol='circle', line=dict(width=2, color='white')),
                        name=f"{trace_data['name']} @ x={vline_x:.6f}",
                        hovertemplate=f"<b>{trace_data['name']}</b><br>{x_param}: {vline_x:.6f}<br>{y_param}: {y_curve[idx]:.6f}<extra></extra>",
                        showlegend=False
                    ))

    if hline_y is not None:
        all_x = [x for t in all_traces_data for x in (t['x_dense'] if t['x_dense'] is not None else t['x'])]
        x_min, x_max = (min(all_x), max(all_x)) if all_x else (0, 1)
        
        fig.add_trace(go.Scatter(
            x=[x_min, x_max], y=[hline_y, hline_y], mode='lines', name='Horizontal Guide',
            line=dict(color='rgba(128, 128, 128, 0.7)', width=1.5, dash='dot'),
            showlegend=False, hoverinfo='skip'
        ))
        
        for trace_data in all_traces_data:
            x_curve = trace_data['x_dense'] if trace_data['x_dense'] is not None else trace_data['x']
            y_curve = trace_data['y_dense'] if trace_data['y_dense'] is not None else trace_data['y']
            
            if len(y_curve) > 0 and y_curve.min() <= hline_y <= y_curve.max():
                idx = np.argmin(np.abs(y_curve - hline_y))
                if abs(y_curve[idx] - hline_y) < (y_curve.max() - y_curve.min()) * 0.05:
                    fig.add_trace(go.Scatter(
                        x=[x_curve[idx]], y=[hline_y], mode='markers',
                        marker=dict(size=12, color=trace_data['color'], symbol='diamond', line=dict(width=2, color='white')),
                        name=f"{trace_data['name']} @ y={hline_y:.6f}",
                        hovertemplate=f"<b>{trace_data['name']}</b><br>{x_param}: {x_curve[idx]:.6f}<br>{y_param}: {hline_y:.6f}<extra></extra>",
                        showlegend=False
                    ))
    
    fig.update_layout(
        title=f"{transistor_name} - {y_param} vs {x_param} {'(with Interpolation)' if enable_interpolation else '(Original Data)'}",
        xaxis_title=x_param, yaxis_title=y_param, template="plotly_white", height=600, width=1150,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode='closest'
    )
    return fig

print("Plotting engine initialized.")

Plotting engine initialized.


In [21]:
# ==========================================
# CELL 4: IPYWIDGETS INTERFACE & DISPLAY
# ==========================================
transistor_dropdown = widgets.Dropdown(options=[('NMOS', 'nmos'), ('PMOS', 'pmos')], value='nmos', description='Transistor:', style={'description_width': '80px'}, layout=widgets.Layout(width='200px'))
y_param_dropdown = widgets.Dropdown(options=available_params, value='gm/gds', description='Y-axis:', style={'description_width': '80px'}, layout=widgets.Layout(width='200px'))
x_param_dropdown = widgets.Dropdown(options=available_params, value='gm/Id', description='X-axis:', style={'description_width': '80px'}, layout=widgets.Layout(width='200px'))

l_checkboxes = widgets.SelectMultiple(
    options=lengths, 
    value=tuple(default_selected_lengths), 
    description='Lengths:', 
    rows=15, 
    style={'description_width': '80px'}, 
    layout=widgets.Layout(width='200px', height='200px')
)

# Guides
vline_checkbox = widgets.Checkbox(value=False, description='X Value:', style={'description_width': '6px'}, layout=widgets.Layout(width='100px'))
vline_fine = widgets.FloatText(value=5.0, step=0.0001, description='', disabled=True, style={'description_width': '0px'}, layout=widgets.Layout(width='110px'))
vline_box = widgets.HBox([vline_checkbox, vline_fine], layout=widgets.Layout(width='200px'))

hline_checkbox = widgets.Checkbox(value=False, description='Y Value:', style={'description_width': '6px'}, layout=widgets.Layout(width='100px'))
hline_fine = widgets.FloatText(value=200.0, step=0.0001, description='', disabled=True, style={'description_width': '0px'}, layout=widgets.Layout(width='110px'))
hline_box = widgets.HBox([hline_checkbox, hline_fine], layout=widgets.Layout(width='200px'))

# Scales and toggles
x_scale_radio = widgets.RadioButtons(options=['linear', 'log'], value='linear', description='X-scale:', style={'description_width': '80px'}, layout=widgets.Layout(width='200px'))
y_scale_radio = widgets.RadioButtons(options=['linear', 'log'], value='linear', description='Y-scale:', style={'description_width': '80px'}, layout=widgets.Layout(width='200px'))
scale_controls = widgets.HBox([x_scale_radio, y_scale_radio], layout=widgets.Layout(width='200px'))

markers_checkbox = widgets.Checkbox(value=False, description='Show Markers', style={'description_width': '20px'}, layout=widgets.Layout(width='200px'))
interpolation_checkbox = widgets.Checkbox(value=True, description='Real Time Interpolation', style={'description_width': '20px'}, layout=widgets.Layout(width='200px'))

plot_output = widgets.Output()

def update_text_ranges():
    data_grouped = nmos_grouped if transistor_dropdown.value == 'nmos' else pmos_grouped
    x_all, y_all = [], []
    for l_val in l_checkboxes.value:
        if l_val in data_grouped:
            x_data = data_grouped[l_val][x_param_dropdown.value].values
            y_data = data_grouped[l_val][y_param_dropdown.value].values
            x_all.extend(x_data[~np.isnan(x_data)])
            y_all.extend(y_data[~np.isnan(y_data)])
    
    if x_all:
        x_min, x_max = min(x_all), max(x_all)
        if vline_fine.value < x_min or vline_fine.value > x_max: vline_fine.value = (x_min + x_max) / 2
    if y_all:
        y_min, y_max = min(y_all), max(y_all)
        if hline_fine.value < y_min or hline_fine.value > y_max: hline_fine.value = (y_min + y_max) / 2

def update_plot(change=None):
    with plot_output:
        clear_output(wait=True)
        update_text_ranges()
        
        vline_fine.disabled = not vline_checkbox.value
        hline_fine.disabled = not hline_checkbox.value
        vline_x = vline_fine.value if vline_checkbox.value else None
        hline_y = hline_fine.value if hline_checkbox.value else None
        
        fig = create_interactive_plot(
            transistor_type=transistor_dropdown.value, y_param=y_param_dropdown.value, x_param=x_param_dropdown.value,
            selected_l_values=list(l_checkboxes.value), show_markers=markers_checkbox.value,
            vline_x=vline_x, hline_y=hline_y, enable_interpolation=interpolation_checkbox.value
        )
        
        fig.update_xaxes(type=x_scale_radio.value)
        fig.update_yaxes(type=y_scale_radio.value)
        display(fig)

# Observers
transistor_dropdown.observe(update_plot, 'value')
y_param_dropdown.observe(update_plot, 'value')
x_param_dropdown.observe(update_plot, 'value')
l_checkboxes.observe(update_plot, 'value')
x_scale_radio.observe(update_plot, 'value')
y_scale_radio.observe(update_plot, 'value')
markers_checkbox.observe(update_plot, 'value')
interpolation_checkbox.observe(update_plot, 'value')
vline_checkbox.observe(update_plot, 'value')
hline_checkbox.observe(update_plot, 'value')
vline_fine.observe(update_plot, 'value')
hline_fine.observe(update_plot, 'value')

# Layout
left_panel = widgets.VBox([
    widgets.HTML("<div style='height: 15px;'></div>"),
    widgets.HTML("<b>Plot Parameters</b>"), transistor_dropdown, y_param_dropdown, x_param_dropdown, l_checkboxes,
    widgets.HTML("<div style='height: 5px;'></div>"),
    widgets.HTML("<b>Guide Lines</b>"), vline_box, hline_box,
    widgets.HTML("<div style='height: 5px;'></div>"),
    widgets.HTML("<b>Display Settings</b>"),
    widgets.HBox([scale_controls], layout=widgets.Layout(justify_content='center', width='100%')),
    widgets.HTML("<div style='height: 5px;'></div>"),
    widgets.HBox([markers_checkbox], layout=widgets.Layout(justify_content='center', width='100%')),
    widgets.HBox([interpolation_checkbox], layout=widgets.Layout(justify_content='center', width='100%'))
], layout=widgets.Layout(width='250px', margin='10px 20px'))

right_panel = widgets.VBox([plot_output], layout=widgets.Layout(width='1400px', margin='0 5px'))
main_layout = widgets.HBox([left_panel, right_panel], layout=widgets.Layout(justify_content='flex-start'))

display(main_layout)
update_plot()